In [ ]:
# 让 notebook 无论从哪个目录启动都能 import retailinsight
import sys
from pathlib import Path

_candidates = [Path.cwd(), *Path.cwd().parents]
_root = next(p for p in _candidates if (p / "retailinsight" / "__init__.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("项目根目录:", _root)


# 05 · 流失预测模型
> 目标：给每个会员一个流失概率，并给出可运营的名单与阈值依据。
> **防泄漏是底线**：特征只用观察窗，标签只看标签窗。

In [ ]:
import json

import pandas as pd

from retailinsight.io.loader import read_parquet
from retailinsight.settings import MART_CHURN, REPORTS_DIR

metrics = json.loads((REPORTS_DIR / "metrics_churn.json").read_text(encoding="utf-8"))
print("backend:", metrics["backend"])
print("时间切分:", metrics["split"])
print("特征数:", metrics["n_features"])
pd.DataFrame({"LR 基线": metrics["lr_baseline"], "GBDT 主模型": metrics["gbdt"],
              "RandomForest": metrics["random_forest"]})


## 5.1 验收线核对

In [ ]:
gb, lr = metrics["gbdt"], metrics["lr_baseline"]
upper = 1 / gb["base_rate"]
print(f"AUC 增益: {gb['auc'] - lr['auc']:+.4f}（验收线 +0.03）→ "
      f"{'达标' if gb['auc'] - lr['auc'] >= 0.03 else '未达标'}")
print(f"KS: {gb['ks']:.4f}（验收线 ≥0.30）→ {'达标' if gb['ks'] >= 0.30 else '未达标'}")
print(f"Lift@20%: {gb['lift_top20']:.2f} / 理论上限 {upper:.2f} = "
      f"{gb['lift_top20'] / upper:.1%}（验收线 ≥85%）")


## 5.2 特征重要性（SHAP 或 permutation importance 降级）

In [ ]:
imp = pd.Series({k: v for k, v in metrics["feature_importance"].items() if k != "method"},
                 dtype="float64").sort_values(ascending=False)
imp.head(15)


## 5.3 验证集上的阈值调优
> 按业务成本最优（漏判 : 误触 = 8 : 1），与《决策建议报告》S2 同口径。

In [ ]:
th = metrics["thresholds"]
print(json.dumps(th, ensure_ascii=False, indent=2))


## 5.4 名单产出与分层

In [ ]:
churn = read_parquet(MART_CHURN)
print(f"打分覆盖 {len(churn):,} 人")
print(churn["risk_level"].value_counts())
print(f"\nTop20 高危名单平均流失概率: {churn.nlargest(int(len(churn)*0.2), 'churn_prob')['churn_prob'].mean():.4f}")
print("\n单会员归因字段:", [c for c in churn.columns if 'shap' in c.lower()])


## 5.5 概率分布

In [ ]:
from retailinsight.viz import charts

print(charts.churn_hist(churn["churn_prob"].to_numpy()))
print(charts.lift_curve(churn["churn"].to_numpy(), churn["churn_prob"].to_numpy()))
